In [4]:
# install BCB package (https://github.com/wilsonfreitas/python-bcb/)
!pip install python-bcb

In [5]:
# bcb csomagból sgs és Expectativas modulok betöltése
from bcb import sgs
from bcb import Expectativas
expect = Expectativas()

import pandas as pd
import matplotlib.pyplot as plt
start_date = '2010-01-01'
end_date = '2018-10-31'

# Árfolyamok
Az értékpapírok árfolyamai: CDI, BRLUS, IBOV, iDkA I3, iDkA P3

A makrogazdasági tényezők: BIR, Selic

In [6]:
# A BCB-ből elérhető értékpapírok adatsorainak betöltése
codes = {
    'CDI': 12,
    'BRLUS': 1,
    'IBOV': 7,
    'Selic': 432,
    'BIR': 433
}

df = sgs.get(codes, start=start_date, end=end_date) # LASSÚ!
df.head(7)

,CDI,BRLUS,IBOV,Selic,BIR
Date,,,,,
2010-01-01,NaN,NaN,NaN,8.75,0.75
2010-01-02,NaN,NaN,NaN,8.75,NaN
2010-01-03,NaN,NaN,NaN,8.75,NaN
2010-01-04,0.032780,1.7240,70045.0,8.75,NaN
2010-01-05,0.032780,1.7227,70239.0,8.75,NaN
2010-01-06,0.032780,1.7337,70729.0,8.75,NaN
2010-01-07,0.032817,1.7413,70451.0,8.75,NaN


Az iDkA adatok beolvasása és hozzáadása, letöltése

In [7]:
#from google.colab import userdata
import requests
import io

# iDkA I3 betöltése
url = 'https://raw.githubusercontent.com/siklerjulianna/TDK-MBL/main/Data_Cantini/IDKAIPCA3A-HISTORICO.xls'
response = requests.get(url)
idi = pd.read_excel(io.BytesIO(response.content), engine='openpyxl')

# iDkA P3 betöltése
url = 'https://raw.githubusercontent.com/siklerjulianna/TDK-MBL/main/Data_Cantini/IDKAPRE3A-HISTORICO.xls'
response = requests.get(url)
idp = pd.read_excel(io.BytesIO(response.content), engine='openpyxl')

# Date oszlop konvertálása
idi['Date'] = pd.to_datetime(idi['Data de Referência'])
idka_i3 = idi[idi['Date'].between(start_date, end_date)]
idka_i3 = idka_i3.rename(columns={'Número Índice': 'iDkA I3'})

idp['Date'] = pd.to_datetime(idp['Data de Referência'])
idka_p3 = idp[idp['Date'].between(start_date, end_date)]
idka_p3 = idka_p3.rename(columns={'Número Índice': 'iDkA P3'})

# Összefűzés
df_merged = pd.merge(df       , idka_i3[['Date', 'iDkA I3']], on='Date', how='left')
df_merged = pd.merge(df_merged, idka_p3[['Date', 'iDkA P3']], on='Date', how='left')
df_merged.set_index('Date', inplace=True)

display(df_merged.head(7))

# Letöltés
from google.colab import files
df_merged.to_csv('securities_factors.csv', index=True)
files.download('securities_factors.csv')

,CDI,BRLUS,IBOV,Selic,BIR,iDkA I3,iDkA P3
Date,,,,,,,
2010-01-01,NaN,NaN,NaN,8.75,0.75,NaN,NaN
2010-01-02,NaN,NaN,NaN,8.75,NaN,NaN,NaN
2010-01-03,NaN,NaN,NaN,8.75,NaN,NaN,NaN
2010-01-04,0.032780,1.7240,70045.0,8.75,NaN,1824.112770,1820.791588
2010-01-05,0.032780,1.7227,70239.0,8.75,NaN,1826.218049,1821.194760
2010-01-06,0.032780,1.7337,70729.0,8.75,NaN,1828.303799,1822.627014
2010-01-07,0.032817,1.7413,70451.0,8.75,NaN,1829.011878,1822.511522


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# A Top 5 előrejelző vállalat előrejelzéseinek mediánja

In [8]:
# factors = ['IPCA', 'Câmbio', 'Selic'] -> ['BIR_exp', 'BRL_exp', 'Sel_exp', 'BIRm_exp']

ep_m = expect.get_endpoint('ExpectativasMercadoTop5Mensais')
ep_s = expect.get_endpoint('ExpectativasMercadoTop5Selic')

dfb = (ep_m.query()
       .filter(ep_m.Indicador == 'IPCA')
       .filter(ep_m.tipoCalculo == 'C')
       .collect())

dfm = (ep_m.query()
       .filter(ep_m.Indicador == 'IPCA')
       .filter(ep_m.tipoCalculo == 'C')
       .collect())

dfc = (ep_m.query()
       .filter(ep_m.Indicador == 'Câmbio')
       .filter(ep_m.tipoCalculo == 'C')
       .collect())

dfs = (ep_s.query()
       .filter(ep_s.indicador == 'Selic')
       .filter(ep_s.tipoCalculo == 'C')
       .collect())

#
def is_next_month(row):
    # Kiszámolja, hogy a Data oszlop alapján mi a következő hónap
    next_month_date = row['Data'] + pd.DateOffset(months=1)
    return (row['DataReferencia'].month == next_month_date.month and
            row['DataReferencia'].year  == next_month_date.year)

# Kiválasztja a legközelebbi várakozást (dfb)
dfb['Data'] = pd.to_datetime(dfb['Data'])
dfb['reuniao'] = pd.to_datetime(dfb['DataReferencia'], format='%m/%Y')
dfb = dfb.query("@start_date <= Data <= @end_date")
idx = dfb.groupby('Data')['reuniao'].idxmin()
dfb = dfb.loc[idx]
dfb = dfb.sort_values('Data')

# Sorok kiszűrése, ahol a Data oszliop év és honap megegyezik a 'reuniao' oszloppal
dfm['Data'] = pd.to_datetime(dfm['Data'])
dfm['reuniao'] = pd.to_datetime(dfm['DataReferencia'], format='%m/%Y')
dfm = dfm.query("@start_date <= Data <= @end_date")
dfm = dfm[
    (dfm['Data'].dt.year == dfm['reuniao'].dt.year) &
    (dfm['Data'].dt.month == dfm['reuniao'].dt.month)
]
dfm = dfm.sort_values('Data')

dfc['Data'] = pd.to_datetime(dfc['Data'])
dfc['reuniao'] = pd.to_datetime(dfc['DataReferencia'], format='%m/%Y')
dfc = dfc.query("@start_date <= Data <= @end_date")
idx = dfc.groupby('Data')['reuniao'].idxmin()
dfc = dfc.loc[idx]
dfc = dfc.sort_values('Data')

# Kiválasztja a legközelebbi várakozást (dfs)
dfs['Data'] = pd.to_datetime(dfs['Data'])
dfs['reuniao'] = pd.to_datetime(dfs['reuniao'], format='R%m/%Y')
dfs = dfs.query("@start_date <= Data <= @end_date")
idx = dfs.groupby('Data')['reuniao'].idxmin()
dfs = dfs.loc[idx]
dfs = dfs.sort_values('Data')

print(dfb.columns)

print(f"BIR_exp:{dfb}")
print(f"BIRm_exp:{dfm}")
print(f"BRL_exp:{dfc}")
print(f"Sel_exp:{dfs}")

# Napi varianciák számítása: szórás^2
dfb['Variance'] = dfb['DesvioPadrao']**2
dfm['Variance'] = dfm['DesvioPadrao']**2
dfc['Variance'] = dfc['DesvioPadrao']**2
dfs['Variance'] = dfs['desvioPadrao']**2

# Mediana átnevezése
dfb = dfb.rename(columns={'Mediana': 'BIR_exp'})
dfm = dfm.rename(columns={'Mediana': 'BIRm_exp'})
dfc = dfc.rename(columns={'Mediana': 'BRL_exp'})
dfs = dfs.rename(columns={'mediana': 'Sel_exp'})

# Variance átnevezése
dfb = dfb.rename(columns={'Variance': 'BIR_var'})
dfm = dfm.rename(columns={'Variance': 'BIRm_var'})
dfc = dfc.rename(columns={'Variance': 'BRL_var'})
dfs = dfs.rename(columns={'Variance': 'Sel_var'})

# Top 5 adatbázis összeillesztése
Top5_merged = pd.merge(dfb, dfc[['Data', 'BRL_exp','BRL_var']], on='Data', how='left')
Top5_merged = pd.merge(Top5_merged, dfs[['Data', 'Sel_exp','Sel_var']], on='Data', how='left')
Top5_merged = pd.merge(Top5_merged, dfm[['Data', 'BIRm_exp','BIRm_var']], on='Data', how='left')
Top5_merged.set_index('Data', inplace=True)

Top5_merged = Top5_merged[['BIR_exp', 'BRL_exp', 'Sel_exp', 'BIRm_exp',
                           'BIR_var', 'BRL_var', 'Sel_var', 'BIRm_var']]

display(Top5_merged.head(7))

# Letöltés
from google.colab import files
Top5_merged.to_csv('top5_expectations.csv', index=True)
files.download('top5_expectations.csv')

Index(['Indicador', 'Data', 'DataReferencia', 'tipoCalculo', 'Media',
       'Mediana', 'DesvioPadrao', 'Minimo', 'Maximo', 'reuniao'],
      dtype='object')
BIR_exp:      Indicador       Data DataReferencia tipoCalculo  Media  Mediana  \
24388      IPCA 2010-01-04        12/2009           C   0.38     0.37   
24402      IPCA 2010-01-05        12/2009           C   0.38     0.37   
24416      IPCA 2010-01-06        12/2009           C   0.38     0.37   
24430      IPCA 2010-01-07        12/2009           C   0.38     0.37   
24444      IPCA 2010-01-08        12/2009           C   0.38     0.37   
...         ...        ...            ...         ...    ...      ...   
67832      IPCA 2018-10-25        10/2018           C   0.59     0.58   
67850      IPCA 2018-10-26        10/2018           C   0.58     0.57   
67868      IPCA 2018-10-29        10/2018           C   0.58     0.57   
67886      IPCA 2018-10-30        10/2018           C   0.58     0.57   
67904      IPCA 2018-10-31     

,BIR_exp,BRL_exp,Sel_exp,BIRm_exp,BIR_var,BRL_var,Sel_var,BIRm_var
Data,,,,,,,,
2010-01-04,0.37,1.74,8.75,0.52,NaN,NaN,NaN,NaN
2010-01-05,0.37,1.74,8.75,0.52,NaN,NaN,NaN,NaN
2010-01-06,0.37,1.74,8.75,0.52,NaN,NaN,NaN,NaN
2010-01-07,0.37,1.74,8.75,0.52,NaN,NaN,NaN,NaN
2010-01-08,0.37,1.74,8.75,0.52,NaN,NaN,NaN,NaN
2010-01-11,0.37,1.74,8.75,0.52,NaN,NaN,NaN,NaN
2010-01-12,0.38,1.74,8.75,0.52,NaN,NaN,NaN,NaN


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>